# Classification Assignment Using Machine Learning Algorithms

**Algorithms:** Support Vector Machine (SVM), Decision Tree, K-Nearest Neighbors (KNN), and Naïve Bayes  
**Dataset:** `global_classification_dataset(1).csv`  
**Target variable:** `Disaster_Severity_Class`

## Aim
To develop and evaluate classification models using SVM, Decision Tree, KNN, and Naïve Bayes to predict disaster severity. The models are evaluated using accuracy, precision, recall, F1-score, and log loss.

## Objectives
1. Load and understand the disaster classification dataset.
2. Perform exploratory data analysis and preprocessing.
3. Split the dataset into training and testing sets.
4. Train four classification algorithms.
5. Evaluate and compare their performance.
6. Identify which algorithm performs best on the selected evaluation metrics.

## How to use this notebook in Google Colab
1. Upload this `.ipynb` file to Google Colab.
2. Upload `global_classification_dataset(1).csv` into the **root of your Google Drive's MyDrive** folder.
3. Run the cells from top to bottom and approve the Google Drive access request.
4. If the CSV is inside a folder, update `DATA_PATH` in the **Connect Google Drive and Load Dataset** cell.

The notebook calculates the evaluation results when you run it; results are not manually hard-coded.

## 1. Introduction

Classification is a supervised machine learning technique used to predict a category or class from input features. In this assignment, the model predicts the severity category of a disaster from the available disaster information.

The target column is `Disaster_Severity_Class`. The dataset contains four severity classes:
- Minor / Localized
- Moderate
- Severe
- Catastrophic

## 2. Dataset Description

The dataset contains 14,644 records and 23 columns. Features include the year and month of occurrence, disaster duration, season, geographical information, disaster type, and several disaster-related flags.

The column `Dis No` is an identifier and is not used as a predictive feature. The column `High_Severity_Flag` is directly derived from the severity class in this dataset, so it is excluded to prevent target leakage (giving the model information that already reveals the answer).

## 3. Importing Required Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss,
    classification_report,
    confusion_matrix
)

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

## 4. Connect Google Drive and Load Dataset
Mount Google Drive and read the CSV file. If you saved the file in a folder, change `DATA_PATH` to the correct path.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Put the CSV in MyDrive, or change this path to its folder location.
DATA_PATH = "/content/drive/MyDrive/global_classification_dataset(1).csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
display(df.head())

## 5. Understanding the Dataset

In [ ]:
print("Number of rows and columns:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nStatistical summary:")
display(df.describe(include="all").T)

## 6. Checking Missing Values and Duplicate Records

In [ ]:
print("Missing values in each column:")
display(df.isnull().sum().to_frame("Missing Values"))

print("Total duplicate rows:", df.duplicated().sum())
print("Unique target classes:", df["Disaster_Severity_Class"].nunique())

## 7. Exploratory Data Analysis (EDA)
### 7.1 Distribution of Disaster Severity

In [ ]:
plt.figure(figsize=(9, 5))
order = df["Disaster_Severity_Class"].value_counts().index
sns.countplot(data=df, x="Disaster_Severity_Class", order=order)
plt.title("Distribution of Disaster Severity Classes")
plt.xlabel("Disaster Severity Class")
plt.ylabel("Number of Records")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

print("Class counts:")
display(df["Disaster_Severity_Class"].value_counts().to_frame("Records"))

print("Class percentages:")
display((df["Disaster_Severity_Class"].value_counts(normalize=True) * 100)
        .round(2).to_frame("Percentage"))

### 7.2 Distribution of Disasters by Year

In [ ]:
plt.figure(figsize=(12, 5))
df["Disaster_Year"].value_counts().sort_index().plot(kind="line", marker="o")
plt.title("Number of Disaster Records by Year")
plt.xlabel("Year")
plt.ylabel("Number of Records")
plt.tight_layout()
plt.show()

### 7.3 Disaster Severity by Disaster Type

In [ ]:
plt.figure(figsize=(12, 6))
severity_by_type = pd.crosstab(
    df["Disaster_Type_Cleaned"],
    df["Disaster_Severity_Class"],
    normalize="index"
)
severity_by_type.plot(kind="bar", stacked=True, figsize=(12, 6))
plt.title("Proportion of Severity Classes by Disaster Type")
plt.xlabel("Disaster Type")
plt.ylabel("Proportion")
plt.xticks(rotation=35, ha="right")
plt.legend(title="Severity Class", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

**EDA Observation:** The class distribution should be checked before model training. If one class has substantially fewer examples than the others, accuracy alone may not describe model performance well. The Catastrophic category is comparatively rare in this dataset, so the per-class classification report and macro F1-score are also useful.

## 8. Data Preprocessing
We separate the target from the input features. We remove the identifier `Dis No` and the leakage column `High_Severity_Flag`. Numerical columns are imputed with the median and standardized. Categorical columns are imputed with the most frequent value and converted to numeric form using One-Hot Encoding.

In [ ]:
TARGET = "Disaster_Severity_Class"

# Remove target, identifier, and the target-derived leakage column.
columns_to_drop = [TARGET]
for col in ["Dis No", "High_Severity_Flag"]:
    if col in df.columns:
        columns_to_drop.append(col)

X = df.drop(columns=columns_to_drop)
y = df[TARGET]

# Identify numeric and categorical columns.
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Target column:", TARGET)
print("Number of numerical features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))
print("Input feature shape:", X.shape)

## 9. Splitting the Dataset
The dataset is divided into 80% training data and 20% testing data. Stratification preserves approximately the same class proportions in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])

print("\nTraining target distribution:")
display(y_train.value_counts().to_frame("Training Records"))

print("\nTesting target distribution:")
display(y_test.value_counts().to_frame("Testing Records"))

## 10. Applying Preprocessing
The preprocessing steps are fitted only on the training data. The same fitted transformations are then applied to the test data, which helps prevent data leakage. Dense output is used here because Gaussian Naïve Bayes requires a dense feature matrix.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

## 11. Model Implementation

### 11.1 Support Vector Machine (SVM)
SVM is a supervised learning algorithm that finds a decision boundary to separate classes. A linear SVM is used here. Probability calibration is added so that predicted probabilities are available for calculating log loss.

In [ ]:
svm_base = LinearSVC(
    C=1.0,
    random_state=RANDOM_STATE,
    max_iter=5000
)

svm_model = CalibratedClassifierCV(
    estimator=svm_base,
    method="sigmoid",
    cv=3
)

svm_model.fit(X_train_processed, y_train)
svm_predictions = svm_model.predict(X_test_processed)

print("SVM model training completed.")

### 11.2 Decision Tree Classifier
A Decision Tree makes predictions by splitting the data into branches based on feature conditions. It is relatively easy to interpret and can model non-linear relationships.

In [ ]:
dt_model = DecisionTreeClassifier(
    criterion="gini",
    max_depth=15,
    min_samples_leaf=5,
    random_state=RANDOM_STATE
)

dt_model.fit(X_train_processed, y_train)
dt_predictions = dt_model.predict(X_test_processed)

print("Decision Tree model training completed.")

### 11.3 K-Nearest Neighbors (KNN)
KNN classifies a record based on the classes of its nearest training examples. Here, the model uses five neighbors. Feature scaling is important for distance-based algorithms, which is why numerical features were standardized.

In [ ]:
knn_model = KNeighborsClassifier(
    n_neighbors=5,
    weights="uniform",
    metric="minkowski",
    p=2
)

knn_model.fit(X_train_processed, y_train)
knn_predictions = knn_model.predict(X_test_processed)

print("KNN model training completed.")

### 11.4 Naïve Bayes Classifier
Gaussian Naïve Bayes is a probabilistic classifier based on Bayes' theorem. It assumes that features are conditionally independent given the class and models numeric feature values using Gaussian distributions.

In [ ]:
nb_model = GaussianNB()

nb_model.fit(X_train_processed, y_train)
nb_predictions = nb_model.predict(X_test_processed)

print("Naive Bayes model training completed.")

## 12. Model Evaluation

The models are evaluated using the following metrics:

- **Accuracy:** Proportion of all test records classified correctly.
- **Precision:** Of the records predicted as a class, how many actually belong to that class.
- **Recall:** Of the records that actually belong to a class, how many were identified correctly.
- **F1-score:** Harmonic mean of precision and recall.
- **Log Loss:** Evaluates the quality of predicted class probabilities; a lower value is better.

Weighted precision, recall, and F1-score are reported so that each class contributes according to its support. Macro F1 is also included to give each class equal weight, which is useful when classes are imbalanced.

In [ ]:
models = {
    "SVM": svm_model,
    "Decision Tree": dt_model,
    "KNN": knn_model,
    "Naive Bayes": nb_model
}

results = []

for model_name, model in models.items():
    y_pred = model.predict(X_test_processed)
    y_prob = model.predict_proba(X_test_processed)

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (Weighted)": precision_score(
            y_test, y_pred, average="weighted", zero_division=0
        ),
        "Recall (Weighted)": recall_score(
            y_test, y_pred, average="weighted", zero_division=0
        ),
        "F1-score (Weighted)": f1_score(
            y_test, y_pred, average="weighted", zero_division=0
        ),
        "F1-score (Macro)": f1_score(
            y_test, y_pred, average="macro", zero_division=0
        ),
        "Log Loss": log_loss(
            y_test, y_prob, labels=model.classes_
        )
    })

results_df = pd.DataFrame(results).sort_values(
    by="F1-score (Weighted)", ascending=False
).reset_index(drop=True)

display(results_df.style.format({
    col: "{:.4f}" for col in results_df.columns if col != "Model"
}))

## 13. Classification Report for Each Model
The classification report shows precision, recall, F1-score, and support separately for every severity class.

In [ ]:
for model_name, model in models.items():
    y_pred = model.predict(X_test_processed)
    print("\n" + "=" * 65)
    print(model_name)
    print("=" * 65)
    print(classification_report(y_test, y_pred, zero_division=0))

## 14. Confusion Matrices
A confusion matrix compares actual classes with predicted classes. Correct predictions appear along the main diagonal; off-diagonal values represent misclassifications.

In [ ]:
class_labels = sorted(y.unique())

for model_name, model in models.items():
    y_pred = model.predict(X_test_processed)
    cm = confusion_matrix(y_test, y_pred, labels=class_labels)

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_labels,
        yticklabels=class_labels
    )
    plt.title(f"{model_name} - Confusion Matrix")
    plt.xlabel("Predicted Class")
    plt.ylabel("Actual Class")
    plt.xticks(rotation=25, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

## 15. Comparative Analysis
The following graph compares the four models on accuracy, weighted precision, weighted recall, and weighted F1-score. Log loss is shown separately because lower values are better and its scale may differ from the other metrics.

In [ ]:
plot_metrics = [
    "Accuracy",
    "Precision (Weighted)",
    "Recall (Weighted)",
    "F1-score (Weighted)"
]

results_plot = results_df.set_index("Model")[plot_metrics]
ax = results_plot.plot(kind="bar", figsize=(11, 6))
ax.set_title("Comparison of Classification Algorithms")
ax.set_xlabel("Machine Learning Algorithm")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(loc="best")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.barplot(data=results_df, x="Model", y="Log Loss")
plt.title("Log Loss Comparison (Lower is Better)")
plt.xlabel("Machine Learning Algorithm")
plt.ylabel("Log Loss")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 16. Best Model Based on Evaluation Metrics
This cell identifies the best model according to weighted F1-score and the model with the lowest log loss. The conclusion is generated from the results produced by your Colab run.

In [ ]:
best_f1_row = results_df.loc[results_df["F1-score (Weighted)"].idxmax()]
best_logloss_row = results_df.loc[results_df["Log Loss"].idxmin()]
best_accuracy_row = results_df.loc[results_df["Accuracy"].idxmax()]

print(
    f"Best model based on weighted F1-score: {best_f1_row['Model']} "
    f"({best_f1_row['F1-score (Weighted)']:.4f})"
)
print(
    f"Best model based on accuracy: {best_accuracy_row['Model']} "
    f"({best_accuracy_row['Accuracy']:.4f})"
)
print(
    f"Best model based on lowest log loss: {best_logloss_row['Model']} "
    f"({best_logloss_row['Log Loss']:.4f})"
)

## 17. Suitability of Each Algorithm

1. **SVM:** Useful when a clear decision boundary can separate classes. It can work well with many features, although training may take longer on larger datasets.
2. **Decision Tree:** Useful when interpretable, rule-based decisions are important. It can model non-linear relationships but may overfit if not controlled.
3. **KNN:** Useful when records with similar feature values are expected to have similar classes. Prediction can be slower as the training dataset grows, and results depend on the distance measure and choice of neighbors.
4. **Naïve Bayes:** Useful for fast probabilistic classification and can be effective when its conditional-independence assumptions are reasonably suitable. Its assumptions may limit performance when features are strongly related.

## 18. Conclusion

In this assignment, four supervised machine learning classification algorithms—SVM, Decision Tree, KNN, and Naïve Bayes—were implemented to predict disaster severity using the disaster dataset.

The dataset was explored and preprocessed, then divided into training and testing sets using an 80:20 split. The models were evaluated using accuracy, precision, recall, F1-score, and log loss. Classification reports and confusion matrices were also used to understand class-wise performance.

The best-performing algorithm should be selected based on the actual evaluation results generated above. For this imbalanced dataset, weighted metrics should be considered together with macro F1-score and the per-class results, especially for the less frequent Catastrophic class. A model with the highest accuracy is not necessarily the best at identifying every severity category.

Further improvement can be explored through hyperparameter tuning, class-imbalance handling, feature selection, and validation on a separate dataset.

## 19. Save the Evaluation Results to Google Drive
This optional cell saves the model comparison table as a CSV file in your Google Drive.

In [ ]:
OUTPUT_PATH = "/content/drive/MyDrive/classification_model_comparison.csv"
results_df.to_csv(OUTPUT_PATH, index=False)
print("Evaluation results saved to:", OUTPUT_PATH)